<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfires
</h2>
</div>

# Module 7B: *Fire Spread Class Balancing*
##### Version Number: 4.0
---
### Contents  
> *Data Preparation*
> *Automatic Class Balancing*
> *Export File*
---
### Notes

**WARNING** this module is computation heavy
- Start with a **baseline model** for comparison.
- Test with multi-classification **tree-based models** (Random Forest, XGBoost) and LGBM.
- Test and evaluate multiple class balancing strategies (No sampling, Oversampling, Undersampling)
- Compare average F1 score among all classes

---
### Inputs
- `X_spread`,`y_spread`,`model_parameters_spread` Reduced or full size Modeling sets

---
### Outputs  

`spread_best_strategy.csv` - dataframe recording best sampling strategy for each method

---
### User Created Dependencies

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_utils import *
from src.model_utils import *
from src.plot_utils import *

---
### Third Party Dependencies

In [2]:
# Core Python libraries
import numpy as np
import pandas as pd
import json
# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing and modeling
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

# Style
sns.set(style='whitegrid')
plt.rcParams["figure.dpi"] = 100

---

In [3]:
X = pd.read_csv('../data/processed/X_spread.csv')
y = pd.read_csv('../data/processed/y_spread.csv').squeeze()  # Load as Series

with open('../data/processed/model_parameters_spread.json', 'r') as f:
    model_parameters_spread = json.load(f)

In [4]:
model_parameters_spread

{'Random Forest': {'n_estimators': 150,
  'max_depth': 20,
  'min_samples_split': 2,
  'max_features': 'sqrt',
  'class_weight': 'balanced'},
 'XGBoost': {'objective': 'multi:softmax',
  'num_class': 5,
  'n_estimators': 200,
  'max_depth': 6,
  'learning_rate': 0.2,
  'verbosity': 0}}

## Prepare Data for Modeling - Scaling, Splitting, and Resampling

In [5]:
# Build tuned models
spread_rf = RandomForestClassifier(**model_parameters_spread['Random Forest'])
spread_xgb = xgb.XGBClassifier(**model_parameters_spread['XGBoost'])

In [6]:
models = {
    'Spread RF' : spread_rf,
    'Spread XGB' : spread_xgb,
}

#### Subset data to save processor

reform = pd.concat([X,y], axis=1)
subset = balancing_subset_df(reform, 'Target_Spread', 10000)

y = subset['Target_Spread']
X = subset.drop(columns='Target_Spread')

In [7]:
# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

## Automatic Class Balancing

To address the extreme inbalance in the dataset, multiple strategies are explored.
- **In Method Balancing** is used when applicable 
- **RandomUnderSampler** will remove random members of the majority class (Low severity) until they are balanced. This creates a much smaller dataset to model.
- **SMOTE (Synthetic Minority Over-sampling Technique)** will generate synthetic members of the minority classes. Introduces potential noise by synthetic sampling

In [8]:
sampling_strategies = ['Undersampling','No_balance','Oversampling']

all_results = []
counter = 1

for strategy in sampling_strategies:
    for name, model in models.items():
        print("running", strategy, counter, "of 6...")
        counter = counter + 1
        result = class_balancing(
            X_train, y_train, X_test, y_test,
            model, model_parameters_spread,
            sampling_strategy=strategy,
            num_classes = 5
        )
        result['Model_Label'] = name
        all_results.append(result)

# Combine into one DataFrame
all_results_df = pd.concat(all_results, axis=0).reset_index(drop=True)

running Undersampling 1 of 6...


running Undersampling 2 of 6...


running No_balance 3 of 6...


running No_balance 4 of 6...


running Oversampling 5 of 6...


running Oversampling 6 of 6...


### Format and display results

In [9]:
strategy_summary = (
    all_results_df[all_results_df['Phase'] == 'Test']
    .groupby(['Balancing','Model_Label'])['F1-Score']
    .mean()
    .reset_index()
    .sort_values(by='F1-Score', ascending=False)
)

strategy_summary_pivot = (
    strategy_summary
    .pivot(index='Model_Label', columns='Balancing', values='F1-Score')
    .sort_values(by='No_balance', ascending=False)
)

strategy_summary_pivot.style.highlight_max(axis=1, color='lightgreen')

Balancing,No_balance,Oversampling,Undersampling
Model_Label,,,
Spread RF,0.909759,0.897749,0.819462
Spread XGB,0.903111,0.897552,0.823094


### Display Best Strategies

In [10]:
# Find best balancing strategy per model
best_strategy = strategy_summary_pivot.idxmax(axis=1)

# Combine with model names into a new DataFrame
best_strategy_df = pd.DataFrame({
    'Model_Label': best_strategy.index,
    'Best_Strategy': best_strategy.values
})

In [11]:
best_strategy_df

,Model_Label,Best_Strategy
0,Spread RF,No_balance
1,Spread XGB,No_balance


### Detailed results (ALL RESULTS)

In [12]:
all_results_df

,Class,Category,Precision,Recall,F1-Score,Phase,Model,Balancing,Model_Label
0,0,0,0.883262,0.770387,0.822848,Train,RandomForestClassifier,Undersampling,Spread RF
1,1,1,0.835426,0.872506,0.853451,Train,RandomForestClassifier,Undersampling,Spread RF
2,2,2,0.901534,0.912365,0.906817,Train,RandomForestClassifier,Undersampling,Spread RF
3,3,3,0.943289,0.964912,0.953951,Train,RandomForestClassifier,Undersampling,Spread RF
4,4,4,0.947604,0.992921,0.969731,Train,RandomForestClassifier,Undersampling,Spread RF
5,0,0,0.983937,0.791425,0.877244,Test,RandomForestClassifier,Undersampling,Spread RF
6,1,1,0.514164,0.880534,0.649229,Test,RandomForestClassifier,Undersampling,Spread RF
7,2,2,0.706543,0.912934,0.796587,Test,RandomForestClassifier,Undersampling,Spread RF
8,3,3,0.805355,0.969622,0.879887,Test,RandomForestClassifier,Undersampling,Spread RF
9,4,4,0.810575,0.997473,0.894364,Test,RandomForestClassifier,Undersampling,Spread RF


---

## Export File

In [13]:
best_strategy_df.to_csv('../data/processed/spread_best_strategy.csv',index=False)
print("All datasets saved successfully to ../data/processed/")

All datasets saved successfully to ../data/processed/
